# Phase 1: Deep Latent Compression (30 Marks)

**Team Astra** | NSSC 2026 | IIT Kharagpur  
**Lead:** 

---

## Objectives
1. Architect a Convolutional / Variational Autoencoder **from scratch** in PyTorch
2. Compress each 227×227 grayscale image into a fixed-length 1D latent vector
3. Design a combined reconstruction loss (MSE + SSIM + KL)
4. Train all 5 architecture versions (v1–v5)
5. Visualize latent space via t-SNE and UMAP

### Key Constraints
- **No pretrained weights** — all models built from scratch
- **No transfer learning** — no VGG, ResNet, or pretrained feature extractors
- **Exact output shape** — decoder must produce exactly (B, 1, 227, 227)

In [ ]:
# ─── System Setup ─────────────────────────────────────────────────────
import sys
import os

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Configuration
IMAGE_DIR = os.path.join(PROJECT_ROOT, 'data', 'images')
METADATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'source_image_metadata.csv')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'models')
PLOTS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'plots')
LATENT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'latent_vectors')

print(f'Project root: {PROJECT_ROOT}')
print(f'Image dir: {IMAGE_DIR}')

In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.utils import set_seed, get_device, ensure_dirs
from src.dataset import MarsHiRISEDataset, create_dataloaders
from src.models import get_model, MODEL_REGISTRY
from src.losses import CombinedLoss
from src.train import train_model
from src.latent_utils import extract_latents, save_latents, plot_tsne, plot_umap
from src.utils import plot_loss_curves, plot_reconstructions

# Reproducibility
set_seed(42)
device = get_device()
ensure_dirs()

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1.1 Data Loading & Exploration

In [ ]:
# Create DataLoaders
train_loader, val_loader, full_loader = create_dataloaders(
    image_dir=IMAGE_DIR,
    batch_size=32,
    val_split=0.1,
    num_workers=4,
    seed=42,
)

# Verify a sample batch
sample_batch = next(iter(train_loader))
images, filenames = sample_batch
print(f'Batch shape: {images.shape}')  # Expected: (32, 1, 227, 227)
print(f'Value range: [{images.min():.3f}, {images.max():.3f}]')
print(f'Sample filenames: {filenames[:3]}')

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    if i < images.shape[0]:
        ax.imshow(images[i].squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
        ax.set_title(filenames[i][:20], fontsize=7)
    ax.axis('off')
fig.suptitle('Sample Mars HiRISE Images (227×227)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'sample_images.png'), dpi=150)
plt.show()

## 1.2 Architecture Verification

Before training, verify that all 5 model versions produce the exact output shape `(B, 1, 227, 227)`.

In [ ]:
# Shape verification for all 5 versions
test_input = torch.randn(2, 1, 227, 227)
print('Architecture Shape Verification')
print('=' * 60)

for version_name, model_class in MODEL_REGISTRY.items():
    model = model_class()
    model.eval()
    with torch.no_grad():
        output = model(test_input)
        if isinstance(output, tuple):
            out_shape = output[0].shape
        else:
            out_shape = output.shape
    
    n_params = sum(p.numel() for p in model.parameters())
    status = '✓' if out_shape == (2, 1, 227, 227) else '✗ MISMATCH'
    print(f'  {version_name.upper():4s} | Output: {str(out_shape):20s} | '
          f'Params: {n_params:>12,} | {status}')
    
    assert out_shape == (2, 1, 227, 227), \
        f'{version_name} shape mismatch: {out_shape}'

print('\n✓ All models verified: exact (B, 1, 227, 227) output')

## 1.3 Training All 5 Versions

### Training Configuration

| Version | Type | Latent | α (MSE) | β (SSIM) | γ (KL) | β-annealing |
|---------|------|--------|---------|----------|--------|-------------|
| v1 | CAE | 128 | 1.0 | 0.0 | 0.0 | No |
| v2 | CAE+BN | 128 | 0.5 | 0.5 | 0.0 | No |
| v3 | VAE | 128 | 0.5 | 0.5 | 0.001 | No |
| v4 | VAE+Skip | 256 | 0.5 | 0.5 | 0.001 | No |
| v5 | β-VAE+Skip | 256 | 0.5 | 0.5 | 0.0 | Yes (max=0.005) |

In [ ]:
# Training configuration for all 5 versions
TRAIN_CONFIGS = {
    'v1': dict(
        epochs=50, lr=1e-3, alpha=1.0, beta=0.0, gamma=0.0,
        patience=10, beta_annealing=False,
    ),
    'v2': dict(
        epochs=50, lr=1e-3, alpha=0.5, beta=0.5, gamma=0.0,
        patience=10, beta_annealing=False,
    ),
    'v3': dict(
        epochs=50, lr=1e-3, alpha=0.5, beta=0.5, gamma=0.001,
        patience=10, beta_annealing=False,
    ),
    'v4': dict(
        epochs=50, lr=1e-3, alpha=0.5, beta=0.5, gamma=0.001,
        patience=10, beta_annealing=False,
    ),
    'v5': dict(
        epochs=50, lr=1e-3, alpha=0.5, beta=0.5, gamma=0.0,
        patience=10, beta_annealing=True, beta_max=0.005,
    ),
}

# Storage for all training histories
all_histories = {}

In [ ]:
# ─── Train v1: Baseline CAE ─────────────────────────────────────────
print('\n' + '█' * 70)
print(' TRAINING v1: Baseline Convolutional Autoencoder')
print('█' * 70)

model_v1 = get_model('v1')
history_v1 = train_model(
    model=model_v1,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    version='v1',
    save_dir=MODELS_DIR,
    **TRAIN_CONFIGS['v1'],
)
all_histories['v1'] = history_v1

plot_loss_curves(
    history_v1['train_total'], history_v1['val_total'],
    title='v1: Baseline CAE — Training Loss',
    save_path=os.path.join(PLOTS_DIR, 'loss_v1.png'),
)

In [ ]:
# ─── Train v2: BatchNorm AE ─────────────────────────────────────────
print('\n' + '█' * 70)
print(' TRAINING v2: BatchNorm Autoencoder')
print('█' * 70)

model_v2 = get_model('v2')
history_v2 = train_model(
    model=model_v2,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    version='v2',
    save_dir=MODELS_DIR,
    **TRAIN_CONFIGS['v2'],
)
all_histories['v2'] = history_v2

plot_loss_curves(
    history_v2['train_total'], history_v2['val_total'],
    title='v2: BatchNorm AE — Training Loss',
    save_path=os.path.join(PLOTS_DIR, 'loss_v2.png'),
    components={'MSE': history_v2['train_mse'], 'SSIM': history_v2['train_ssim']},
)

In [ ]:
# ─── Train v3: Variational Autoencoder ──────────────────────────────
print('\n' + '█' * 70)
print(' TRAINING v3: Variational Autoencoder')
print('█' * 70)

model_v3 = get_model('v3')
history_v3 = train_model(
    model=model_v3,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    version='v3',
    save_dir=MODELS_DIR,
    **TRAIN_CONFIGS['v3'],
)
all_histories['v3'] = history_v3

plot_loss_curves(
    history_v3['train_total'], history_v3['val_total'],
    title='v3: VAE — Training Loss',
    save_path=os.path.join(PLOTS_DIR, 'loss_v3.png'),
    components={
        'MSE': history_v3['train_mse'],
        'SSIM': history_v3['train_ssim'],
        'KL': history_v3['train_kl'],
    },
)

In [ ]:
# ─── Train v4: VAE + Skip Connections ───────────────────────────────
print('\n' + '█' * 70)
print(' TRAINING v4: VAE + Skip Connections (latent_dim=256)')
print('█' * 70)

model_v4 = get_model('v4')
history_v4 = train_model(
    model=model_v4,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    version='v4',
    save_dir=MODELS_DIR,
    **TRAIN_CONFIGS['v4'],
)
all_histories['v4'] = history_v4

plot_loss_curves(
    history_v4['train_total'], history_v4['val_total'],
    title='v4: VAE + Skip — Training Loss',
    save_path=os.path.join(PLOTS_DIR, 'loss_v4.png'),
    components={
        'MSE': history_v4['train_mse'],
        'SSIM': history_v4['train_ssim'],
        'KL': history_v4['train_kl'],
    },
)

In [ ]:
# ─── Train v5: β-VAE + Skip + LeakyReLU ─────────────────────────────
print('\n' + '█' * 70)
print(' TRAINING v5: β-VAE + Skip + LeakyReLU (β-annealing)')
print('█' * 70)

model_v5 = get_model('v5')
history_v5 = train_model(
    model=model_v5,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    version='v5',
    save_dir=MODELS_DIR,
    **TRAIN_CONFIGS['v5'],
)
all_histories['v5'] = history_v5

plot_loss_curves(
    history_v5['train_total'], history_v5['val_total'],
    title='v5: β-VAE Final — Training Loss',
    save_path=os.path.join(PLOTS_DIR, 'loss_v5.png'),
    components={
        'MSE': history_v5['train_mse'],
        'SSIM': history_v5['train_ssim'],
        'KL': history_v5['train_kl'],
    },
)

## 1.4 Reconstruction Quality Comparison

In [ ]:
# Visualize reconstructions from all 5 versions on the same batch
sample_images = next(iter(val_loader))[0][:8].to(device)

models = {
    'v1': model_v1, 'v2': model_v2, 'v3': model_v3,
    'v4': model_v4, 'v5': model_v5,
}

for version, model in models.items():
    model.eval()
    with torch.no_grad():
        output = model(sample_images)
        recon = output[0] if isinstance(output, tuple) else output
    
    plot_reconstructions(
        sample_images, recon, n=8,
        title=f'{version.upper()} Reconstructions',
        save_path=os.path.join(PLOTS_DIR, f'recon_{version}.png'),
    )

## 1.5 Latent Vector Extraction & Visualization

In [ ]:
# Extract and save latent vectors for the best model (v5)
# Also extract for v1 and v3 for comparison

for version, model in [('v1', model_v1), ('v3', model_v3), ('v5', model_v5)]:
    print(f'\n--- Extracting latents for {version.upper()} ---')
    model.to(device)
    latents, filenames = extract_latents(model, full_loader, device)
    save_latents(latents, filenames, save_dir=LATENT_DIR, version=version)
    
    # t-SNE visualization
    plot_tsne(
        latents, perplexity=30,
        title=f'{version.upper()} — t-SNE Latent Space',
        save_path=os.path.join(PLOTS_DIR, f'tsne_{version}.png'),
    )
    
    # UMAP visualization
    plot_umap(
        latents, n_neighbors=15, min_dist=0.1,
        title=f'{version.upper()} — UMAP Latent Space',
        save_path=os.path.join(PLOTS_DIR, f'umap_{version}.png'),
    )

## 1.6 Comparative Loss Summary

In [ ]:
# Summary table of final losses across all versions
import pandas as pd

summary_rows = []
for version in ['v1', 'v2', 'v3', 'v4', 'v5']:
    h = all_histories[version]
    summary_rows.append({
        'Version': version.upper(),
        'Best Val Loss': min(h['val_total']),
        'Final Val MSE': h['val_mse'][-1],
        'Final Val SSIM': h['val_ssim'][-1],
        'Final Val KL': h['val_kl'][-1],
        'Epochs Trained': len(h['train_total']),
    })

summary_df = pd.DataFrame(summary_rows)
print('\nTraining Summary — All 5 Versions')
print('=' * 70)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(PLOTS_DIR, 'training_summary.csv'), index=False)

In [ ]:
# Comparative loss curves
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
colors = {'v1': '#e74c3c', 'v2': '#2ecc71', 'v3': '#3498db', 'v4': '#f39c12', 'v5': '#9b59b6'}

for version in ['v1', 'v2', 'v3', 'v4', 'v5']:
    h = all_histories[version]
    ax.plot(h['val_total'], color=colors[version], linewidth=2,
            label=f'{version.upper()} (best={min(h["val_total"]):.4f})')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Loss', fontsize=12)
ax.set_title('Comparative Validation Loss — All 5 Versions', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'comparative_loss.png'), dpi=150)
plt.show()

print('\n✓ Phase 1 complete. Latent vectors saved for Phase 2.')